In [1]:
from nnsight import LanguageModel
from nnsight.models.UnifiedTransformer import UnifiedTransformer

import sys

sys.path.append("../../../")

from nngine import alter

import torch 
import einops

tl_model = UnifiedTransformer(
    'gpt2-small',
    processing=False,
    center_writing_weights=False,
    center_unembed=False,
    fold_ln=False,
    device="cuda",
)

tl_model.set_use_hook_mlp_in(True)
tl_model.set_use_split_qkv_input(True)
tl_model.set_use_attn_result(True)

tokenizer = tl_model.tokenizer

model = LanguageModel("openai-community/gpt2", device_map="auto", dispatch=True)

def test_resolution(ground_truth, sample):
    for resolution in [1e-12, 1e-10, 1e-6, 1e-4, 1e-3, 1e-2, 1.0]:
        pct = (sample - ground_truth > resolution).float().mean().item()
        print(f'pct out of range {resolution:.1e} = {pct:.2%}')

Loaded pretrained model gpt2-small into HookedTransformer


In [5]:
import torch 

clean = "When Alan and Alex got a drink at the store, Alex decided to give it to"
corrupted = "When Alan and Alex got a drink at the store, Alan decided to give it to"

clean = tokenizer.encode(clean)
corrupted = tokenizer.encode(corrupted)

with torch.no_grad():
    out = model.trace(clean, trace=False)
    out = out.logits[:,-1,:]

def metric(corr_logits, clean_logits=out):
    return (clean_logits - corr_logits).mean()

In [26]:
with model.trace(corrupted):

    test = model.transformer.h[8].mlp.output.grad.save()


    logits = model.output.logits.save()
    value = metric(logits)
    value.backward()

with tl_model.trace(corrupted):
    tl_test = tl_model.blocks[8].mlp.output.grad.save()

    tl_logits = tl_model.unembed.output.save()
    
    value = metric(tl_logits)
    value.backward()

In [27]:
test_resolution(tl_test, test)
print("LOGITS")
test_resolution(tl_logits, logits)

pct out of range 1.0e-12 = 48.67%
pct out of range 1.0e-10 = 48.67%
pct out of range 1.0e-06 = 0.00%
pct out of range 1.0e-04 = 0.00%
pct out of range 1.0e-03 = 0.00%
pct out of range 1.0e-02 = 0.00%
pct out of range 1.0e+00 = 0.00%
LOGITS
pct out of range 1.0e-12 = 36.56%
pct out of range 1.0e-10 = 36.56%
pct out of range 1.0e-06 = 36.56%
pct out of range 1.0e-04 = 1.14%
pct out of range 1.0e-03 = 0.00%
pct out of range 1.0e-02 = 0.00%
pct out of range 1.0e+00 = 0.00%


In [29]:
model = LanguageModel("openai-community/gpt2", device_map="auto", dispatch=True)

alter(model)

with model.trace(corrupted):
    
    for i in range(7):
        _ = model.transformer.layers[i].attn.split_q.input

    test = model.transformer.layers[8].mlp.output.grad.save()

    logits = model.output.logits.save()
    value = metric(logits)
    value.backward()

with tl_model.trace(corrupted):
    tl_test = tl_model.blocks[8].mlp.output.grad.save()

    tl_logits = tl_model.unembed.output.save()
    
    value = metric(tl_logits)
    value.backward()

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [30]:
test_resolution(test, tl_test)

pct out of range 1.0e-12 = 49.41%
pct out of range 1.0e-10 = 49.39%
pct out of range 1.0e-06 = 0.00%
pct out of range 1.0e-04 = 0.00%
pct out of range 1.0e-03 = 0.00%
pct out of range 1.0e-02 = 0.00%
pct out of range 1.0e+00 = 0.00%


In [ ]:
model = LanguageModel("openai-community/gpt2", device_map="auto", dispatch=True)

alter(model)

with model.trace(sample):

    test = model.transformer.layers[0].attn.split_q.input.grad.save()

    logits = model.output.logits[:,-1,:]
    value = logits.sum()
    value.backward()

with tl_model.trace(sample):
    tl_test = tl_model.blocks[0].hook_q_input.input[0][0].grad.save()

    logits = tl_model.unembed.output[:,-1,:]
    value = logits.sum()
    value.backward()

In [ ]:
test_resolution(test, tl_test)

In [ ]:
with model.trace(sample):

    test = model.transformer.layers[0].attn.attn_result.output.save()

with tl_model.trace(sample):
    tl_test = tl_model.blocks[0].attn.hook_result.output.save()


In [ ]:
print(test.shape)
print(tl_test.shape)

test_resolution(tl_test,test)